# Amazon Prime TV Shows and Movies — Exploratory Data Analysis
**Project type:** EDA + Classification | **Contribution:** Individual

## Project summary
This project studies the Amazon Prime catalogue represented by `titles.csv` and the associated people/role records in `credits.csv`. The two datasets are joined using the common title identifier `id`. Since credits contain multiple rows per title, they are first aggregated into actor and director counts to avoid multiplying title records during the merge.

The analysis explores catalogue composition, release trends, runtime, seasons, age certification, genres, production countries, IMDb/TMDb ratings and popularity. It uses univariate, bivariate and multivariate visualisations, then applies statistical tests to selected comparisons. The accompanying ML notebook builds a baseline classifier to identify titles with IMDb ratings of at least 7.0. This is an analytical exercise on catalogue metadata, not a claim that ratings can be guaranteed or that the model predicts audience behaviour causally.

## Problem statement and business objective
**Problem:** Understand the content catalogue and identify metadata patterns associated with highly rated titles.

**Business objective:** Provide data-backed catalogue insights that can support content discovery, catalogue curation and further recommendation-system development. The analysis is descriptive and should not be interpreted as evidence of customer demand or streaming performance.

## 1. Import libraries and load the two datasets

In [ ]:
# Google Colab: upload titles.csv and credits.csv when prompted.
from google.colab import files
import os, ast, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
sns.set_theme(style="whitegrid")
if not (os.path.exists("/content/titles.csv") and os.path.exists("/content/credits.csv")):
    uploaded = files.upload()
titles_path = "/content/titles.csv"
credits_path = "/content/credits.csv"
titles = pd.read_csv(titles_path, low_memory=False)
credits = pd.read_csv(credits_path, low_memory=False)
print("titles:", titles.shape, "credits:", credits.shape)
display(titles.head())
display(credits.head())

## 2. Data understanding and merge

In [ ]:
# Credits is one-to-many. Aggregate first so the merge retains one row per title.
credits["role_norm"] = credits["role"].astype(str).str.upper()
credit_summary = credits.groupby("id").agg(
    actor_count=("role_norm", lambda s: int((s == "ACTOR").sum())),
    director_count=("role_norm", lambda s: int((s == "DIRECTOR").sum()))
).reset_index()
df = titles.merge(credit_summary, on="id", how="left", validate="one_to_one")
df[["actor_count", "director_count"]] = df[["actor_count", "director_count"]].fillna(0).astype(int)
def parse_list(value):
    try:
        parsed = ast.literal_eval(value) if isinstance(value, str) else []
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []
df["genres_list"] = df["genres"].apply(parse_list)
df["countries_list"] = df["production_countries"].apply(parse_list)
df["genre_count"] = df["genres_list"].str.len()
df["country_count"] = df["countries_list"].str.len()
print("Merged shape:", df.shape)
print("Unique title IDs:", df["id"].nunique())
display(df.head())

In [ ]:
print("Rows and columns:", df.shape)
display(df.info())
display(df.describe(include="all").T)
print("Duplicate full rows:", df.duplicated().sum())
print("Duplicate title IDs:", df["id"].duplicated().sum())
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))
plt.figure(figsize=(12,5)); missing[missing>0].plot(kind="bar"); plt.title("Missing values by column"); plt.ylabel("Count"); plt.tight_layout(); plt.show()

## 3. Data wrangling

- Credits are grouped by title ID before merging; this prevents duplicate title rows caused by multiple cast/crew records.
- Missing credit counts are filled with zero because no matching role rows were found.
- Genre and production-country string representations are parsed into Python lists for analysis.
- Missing IMDb scores are retained for EDA but excluded from supervised model training.
- `seasons` is naturally missing for most movies; it is not globally filled with zero in the raw analysis table.

### Chart 1: Catalogue type distribution
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** The catalogue contains both movies and shows; the bars show their relative representation.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
df['type'].value_counts().plot(kind='bar')
plt.title('Catalogue type distribution')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 2: Titles by release year
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** The time series reveals catalogue coverage across release years and any concentration in recent or older content.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
df['release_year'].value_counts().sort_index().plot(figsize=(12,4))
plt.title('Titles by release year')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 3: IMDb rating distribution
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** The histogram shows the distribution and spread of available IMDb ratings.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
df['imdb_score'].dropna().plot(kind='hist',bins=25,figsize=(8,4))
plt.title('IMDb rating distribution')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 4: Runtime distribution
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** Runtime distribution highlights common content lengths and unusually long/short entries.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
df['runtime'].plot(kind='hist',bins=30,figsize=(8,4))
plt.title('Runtime distribution')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 5: Age certification counts
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** Certification counts help describe the audience-rating mix; missing certification is shown explicitly.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
df['age_certification'].fillna('Not Rated').value_counts().head(12).plot(kind='bar',figsize=(9,4))
plt.title('Age certification counts')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 6: IMDb rating by content type
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** The boxplot compares rating medians and spread for movies and shows.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df,x='type',y='imdb_score')
plt.title('IMDb rating by content type')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 7: Runtime by content type
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** This comparison shows the different runtime profiles of movies and shows.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df,x='type',y='runtime')
plt.title('Runtime by content type')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 8: IMDb votes vs IMDb score
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** The scatterplot explores whether vote volume is associated with rating; association is not causation.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=df.sample(min(2500,len(df)),random_state=42),x='imdb_votes',y='imdb_score',alpha=.45)
plt.title('IMDb votes vs IMDb score')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 9: TMDb score vs IMDb score
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** The two rating systems can be compared for consistency among titles with both scores.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=df,x='tmdb_score',y='imdb_score',alpha=.35)
plt.title('TMDb score vs IMDb score')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 10: TMDb popularity distribution
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** Popularity is often skewed; the histogram shows its range and concentration.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
df['tmdb_popularity'].dropna().plot(kind='hist',bins=30,figsize=(8,4))
plt.title('TMDb popularity distribution')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 11: Top genres
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** The most frequent genre labels indicate catalogue breadth and dominant categories.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
pd.Series([g for row in df.genres_list for g in row]).value_counts().head(15).sort_values().plot(kind='barh',figsize=(8,6))
plt.title('Top genres')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 12: Top production countries
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** Production-country frequency shows the geographic distribution represented in the catalogue.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
pd.Series([c for row in df.countries_list for c in row]).value_counts().head(15).sort_values().plot(kind='barh',figsize=(8,6))
plt.title('Top production countries')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 13: Average IMDb score by release decade
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** Decade-level averages smooth year-to-year variation and show historical rating patterns.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(data=df.dropna(subset=['imdb_score']).assign(decade=lambda x:(x.release_year//10)*10).groupby('decade',as_index=False).imdb_score.mean(),x='decade',y='imdb_score',marker='o')
plt.title('Average IMDb score by release decade')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 14: Average runtime by release decade
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** This chart explores whether typical content duration changes across decades.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(data=df.assign(decade=lambda x:(x.release_year//10)*10).groupby('decade',as_index=False).runtime.mean(),x='decade',y='runtime',marker='o')
plt.title('Average runtime by release decade')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 15: Actor count vs IMDb score
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** This explores whether cast-record counts show any relationship with IMDb score.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(data=df.sample(min(2500,len(df)),random_state=7),x='actor_count',y='imdb_score',alpha=.4)
plt.title('Actor count vs IMDb score')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 16: Director count vs IMDb score
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** Grouping director counts makes the relationship easier to compare while retaining small groups.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df.assign(director_band=pd.cut(df.director_count,[-1,0,1,3,100],labels=['0','1','2–3','4+'])),x='director_band',y='imdb_score')
plt.title('Director count vs IMDb score')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 17: Genre count vs IMDb score
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** The chart compares rating distributions by the number of genre tags.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df,x='genre_count',y='imdb_score')
plt.title('Genre count vs IMDb score')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 18: Country count vs IMDb score
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** The chart explores whether international co-production counts are associated with rating.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df,x='country_count',y='imdb_score')
plt.title('Country count vs IMDb score')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 19: Correlation heatmap
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** The heatmap summarises linear correlations among numeric variables; correlation does not imply causation.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
sns.heatmap(df[['release_year','runtime','seasons','imdb_score','imdb_votes','tmdb_popularity','tmdb_score','actor_count','director_count','genre_count','country_count']].corr(numeric_only=True),annot=True,cmap='coolwarm',fmt='.2f')
plt.title('Correlation heatmap')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### Chart 20: Multivariate rating by type and certification
**Why this chart?** It is suitable for comparing the distribution or relationship described by its title.

**Insight to discuss after execution:** This chart compares ratings across certification groups and content types; sparse groups should be interpreted cautiously.

**Business relevance:** Use the observed pattern to guide catalogue exploration, but validate it against customer engagement and viewing data before making commercial decisions.

In [ ]:
plt.figure(figsize=(10,5))
sns.boxplot(data=df.dropna(subset=['imdb_score']).assign(cert=df.age_certification.fillna('Not Rated')),x='cert',y='imdb_score',hue='type')
plt.title('Multivariate rating by type and certification')
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

## 4. Hypothesis testing

Use only records with non-null IMDb scores. The tests below use a 5% significance level. Report p-values and avoid interpreting statistical significance as business causality.

In [ ]:
rated = df.dropna(subset=["imdb_score"]).copy()
# H1: Mean IMDb scores differ between movies and shows (Welch independent t-test).
movie = rated.loc[rated.type == "MOVIE", "imdb_score"]
show = rated.loc[rated.type == "SHOW", "imdb_score"]
t1 = stats.ttest_ind(movie, show, equal_var=False, nan_policy="omit")
print("H1: Movie vs show mean rating | statistic:", t1.statistic, "p-value:", t1.pvalue)
# H2: IMDb and TMDb scores are positively associated (Spearman correlation).
paired = rated[["imdb_score","tmdb_score"]].dropna()
t2 = stats.spearmanr(paired.imdb_score, paired.tmdb_score)
print("H2: IMDb vs TMDb score | rho:", t2.statistic, "p-value:", t2.pvalue)
# H3: Rating distributions differ between high- and low-popularity halves (Mann–Whitney U).
pop = rated.dropna(subset=["tmdb_popularity"])
median_pop = pop.tmdb_popularity.median()
high_pop = pop.loc[pop.tmdb_popularity >= median_pop, "imdb_score"]
low_pop = pop.loc[pop.tmdb_popularity < median_pop, "imdb_score"]
t3 = stats.mannwhitneyu(high_pop, low_pop, alternative="two-sided")
print("H3: High vs low TMDb popularity | statistic:", t3.statistic, "p-value:", t3.pvalue)
alpha=0.05
for label, result in [("H1",t1),("H2",t2),("H3",t3)]:
    print(label, "Reject H0" if result.pvalue < alpha else "Fail to reject H0")

## 5. EDA conclusion
Summarise the strongest patterns visible in the executed charts and hypothesis tests. Do not insert assumed values: use the notebook outputs. Recommended next step is to combine catalogue metadata with actual viewing, search, completion and subscription data before making recommendation or acquisition decisions.